# Rollback to the Previous Revision

![Agent rollback](imgs/agent_rollback.png)

## Objective
If a freshly promoted revision regresses, roll back by shifting traffic to the previous revision. Method is near-instant and safe to automate because it only ever restores an already-approved revision. This protects staging; production promotion stays a manual gate.

Two methods demonstrated:
1. **Automated rollback** — described, not built: WIP wire the online monitor to trigger this on a canary regression.
2. **Manual rollback** — runnable: list revisions and shift 100% of traffic back to the prior one.

## Method 1 — Automated Rollback (WIP)

The hands-off version mirrors NB3's close-the-loop, but scoped to the new (canary) revision on a short window, so a freshly-shipped dud trips it within its bake:
- The Online Monitor samples the agent's Final Response Quality labeled by revision.
- A canary-scoped alert (short `alignmentPeriod`, filtered to the new revision) fires when that revision's rolling mean drops below the drift threshold.
- That alert triggers the rollback in Method 2 (shift traffic to the previous revision).

The trigger plumbing is the same wiring as NB3's hands-off trigger (alert → Pub/Sub → function), just pointed at a separate canary alert. The detection source is the Online Monitor.

Documentation: [Online monitoring](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/online-monitoring) · [Manage revisions and traffic](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/manage-revisions-and-traffic)

## Method 2 — Manual Rollback to the Previous Revision

Because the agent deploys via the source/container path, each promotion registers a runtime revision. You can revert by shifting 100% of traffic to the prior revision, no redeploy. Needs ≥ 2 revisions (e.g. NB1 create + NB3 promotion).

In [1]:
import yaml
import pandas as pd
import vertexai
from vertexai import agent_engines
from google.genai import types as genai_types
from IPython.display import display

PROJECT = 'sandbox-401718'  # @param {type:"string"}
REGION  = 'us-central1'     # @param {type:"string"}
AGENT   = 'brand-guidelines-checker'
AGENT_DIR = f'../agents/{AGENT}'

vertexai.init(project=PROJECT, location=REGION)
with open(f'{AGENT_DIR}/agent.yaml') as f:
    _name = yaml.safe_load(f).get('display_name') or AGENT
engine = next((e for e in agent_engines.list()
               if (getattr(e, 'display_name', '') or '') == _name), None)
assert engine is not None, f"no deployed engine named {_name!r} — run NB1/NB3 first"
client = vertexai.Client(project=PROJECT, location=REGION,
                         http_options=genai_types.HttpOptions(api_version='v1beta1'))
print('engine:', engine.resource_name)

/opt/conda/lib/python3.10/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


engine: projects/757654702990/locations/us-central1/reasoningEngines/3538275147427348480


In [2]:
# List revisions (newest last). The list returns wrappers — fields live on r.api_resource.
def _ar(r):
    return getattr(r, 'api_resource', None) or r

revs = sorted(client.agent_engines.runtimes.revisions.list(name=engine.resource_name),
              key=lambda r: str(_ar(r).create_time))
display(pd.DataFrame([{'revision': (_ar(r).name or '').split('/')[-1],
                       'state': str(_ar(r).state),
                       'created': str(_ar(r).create_time)} for r in revs]))

assert len(revs) >= 2, "need >=2 revisions to roll back (deploy in NB1, then promote in NB3)"
current, previous = _ar(revs[-1]), _ar(revs[-2])
print('current   :', current.name.split('/')[-1])
print('rollback ->', previous.name.split('/')[-1], '(previous)')

,revision,state,created
0,1,None,2026-06-03 20:31:36.040790+00:00
1,2,None,2026-06-03 21:00:59.276371+00:00


current   : 2
rollback -> 1 (previous)


In [4]:
# Shift 100% of traffic to the previous revision (the rollback). Safe — it only restores an
# already-approved revision, never ships anything new. trafficSplitManual is the documented
# v1beta1 op; verify it on your first real rollback. Mirror script: deployment/rollback.py.

DO_ROLLBACK = True  # @param {type:"boolean"}  — set True to actually shift traffic #######

if DO_ROLLBACK:
    client.agent_engines.update(
        name=engine.resource_name,
        config={'traffic_config': {'trafficSplitManual': {
            'targets': [{'runtimeRevisionName': previous.name, 'percent': 100}]}}})
    print('rolled back: 100% traffic >', previous.name.split('/')[-1])
else:
    print('dry run — set DO_ROLLBACK=True to shift 100% traffic to',
          previous.name.split('/')[-1])

rolled back: 100% traffic -> 1
